In [ ]:
import urllib.request
from pathlib import Path
import load_gml
import itertools

import asyncio
import aiohttp
from tqdm import tqdm

In [ ]:
FOLDER = Path('data/gml')
sorted_fnames = reversed(sorted(FOLDER.glob('*.json'), key=lambda x: int(x.with_suffix('').name)))
katsu_fnames = load_gml.filter_by_application(sorted_fnames)
# katsu_fnames = list(itertools.islice(katsu_fnames, 3000))
# katsu_fnames = list(itertools.islice(katsu_fnames, 3000, 9000))
katsu_fnames = list(itertools.islice(katsu_fnames, 9000, None))
print(katsu_fnames[:10])
print(len(katsu_fnames))

In [ ]:
url_template = 'https://000000book.com/system/images/{a:03d}/{b:03d}/{c:03d}/original/upload.jpg'
id2url = {}
for fname in katsu_fnames:
    n = int(fname.with_suffix('').name)
    first_3 = (n // 1000000) % 1000000
    next_3 = (n // 1000) % 1000
    last_3 = n % 1000
    img_url = url_template.format(a=first_3, b=next_3, c=last_3)
    # urllib.request.urlretrieve(img_url, f'./data/gml_images/{n}.jpg')
    id2url[n] = img_url


In [ ]:
async def download_image(session, semaphore, id, url, output_dir='./data/gml_images/'):
    async with semaphore:
        async with session.get(url) as response:
            if response.status == 200:
                with open(f'{output_dir}/{id}.jpg', 'wb') as f:
                    while True:
                        chunk = await response.content.read(1024)
                        if not chunk:
                            break
                        f.write(chunk)

async def download_all_images(id2url, max_concurrent=20):
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = []
    async with aiohttp.ClientSession() as session:
        for id, url in id2url.items():
            tasks.append(download_image(session, semaphore, id, url))
        
        # Use tqdm to track progress
        for future in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            await future

# Run the asyncio event loop in Jupyter notebook
await download_all_images(id2url)